# Air France India Job Scraper
## India Jobs Scraper (v2.1 - March 2026)
**Source:** recrutement.airfrance.com (Custom ASP.NET portal)

**ATS:** Custom — HTML pagination via ?page=N

**Note:** Air France posts ~86 global jobs; India roles are filtered post-scrape by location.

In [1]:
!pip install selenium webdriver-manager pandas openpyxl requests beautifulsoup4 lxml -q


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import sys, time, random, re
from pathlib import Path
SCRIPTS_DIR = Path.home() / 'Job_Scrapers' / 'All_Scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
from scraper_utils import *
from bs4 import BeautifulSoup
from datetime import datetime
import requests
LOCATION_FILTER = ''
print('Imports loaded. Date:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

scraper_utils.py loaded successfully
Schema: 25 columns
Skills DB: 79 skills
Imports loaded. Date: 2026-03-31 21:04:46


In [3]:
COMPANY = 'Air_France'
OUTPUT_DIR = get_output_dir(COMPANY)
print(f'Output directory: {OUTPUT_DIR}')

Output directory: /Users/incognito/Job_Scrapers/All_CSV_Outputs/Air_France/Outputs/2026_03_31


In [4]:
print('=' * 60)
print('AIR FRANCE JOB SCRAPER')
print('Portal: recrutement.airfrance.com (Custom ASP.NET)')
print('=' * 60)

BASE_URL = 'https://recrutement.airfrance.com'
LIST_URL = f'{BASE_URL}/job/list-of-all-jobs.aspx'

session = get_session()
session.headers.update({'Accept': 'text/html,application/xhtml+xml,*/*', 'Referer': BASE_URL})

airfrance_jobs = []
seen_ids = set()


def parse_jobs_page(soup):
    jobs = []
    # Air France uses <dl> or <ul> job listing structure
    cards = (soup.select('dl.job-item') or soup.select('[class*="job-item"]') or
             soup.select('li[class*="job"]') or soup.select('[class*="offer"]'))
    if not cards:
        # Fallback: find job links with specific URL pattern
        job_links = soup.select('a[href*="emploi-"], a[href*="offre-de-emploi"], a[href*="/job/"]')
        seen = set()
        for link in job_links:
            p = link.find_parent(['dl', 'li', 'div', 'tr'])
            if p and id(p) not in seen: cards.append(p); seen.add(id(p))
    for card in cards:
        title_el = card.select_one('h3 a') or card.select_one('h2 a') or card.select_one('a[href*="emploi"]')
        title = title_el.get_text(strip=True) if title_el else ''
        if not is_valid_job_title(title): continue
        href = title_el.get('href', '') if title_el else ''
        job_url = href if href.startswith('http') else (BASE_URL + href if href else '')
        # Ref number from text like "2025-22566" or from URL
        ref_match = re.search(r'(\d{4}-\d+)', card.get_text())
        job_id = ref_match.group(1) if ref_match else href.rstrip('/').split('/')[-1].split('.')[0]
        if not job_id: job_id = str(abs(hash(title + job_url)))
        loc_el = card.select_one('[class*="location"]') or card.select_one('[class*="lieu"]')
        loc = loc_el.get_text(strip=True) if loc_el else ''
        # Contract type
        contract_el = card.select_one('[class*="contract"]') or card.select_one('[class*="contrat"]')
        contract = contract_el.get_text(strip=True) if contract_el else ''
        emp_type = 'full-time'
        if 'stage' in contract.lower() or 'intern' in title.lower(): emp_type = 'internship'
        elif 'cdd' in contract.lower() or 'temp' in contract.lower(): emp_type = 'contract'
        dept_el = card.select_one('[class*="category"]') or card.select_one('[class*="domain"]')
        dept = dept_el.get_text(strip=True) if dept_el else ''
        jobs.append({
            'job_id': str(job_id), 'title': title, 'company_name': 'Air France',
            'job_url': job_url, 'source_api_url': LIST_URL,
            'business_unit': dept, 'raw_jd_text': card.get_text(' ', strip=True),
            'location_city': loc.split(',')[0].strip() if loc else 'France',
            'location_country': 'France',
            'industry': 'Aviation / Transportation',
            'date_posted': datetime.now().strftime('%Y-%m-%d'),
            'employment_type': emp_type,
            'is_active': True, 'salary_currency': 'EUR', 'source_platform': 'Air France Custom'
        })
    return jobs


# Requests-based scraping (ASP.NET renders server-side)
page = 1
consecutive_empty = 0
while page <= 20:
    params = {'page': page, 'LCID': '2057'}  # LCID=2057 = English
    try:
        resp = session.get(LIST_URL, params=params, timeout=30)
        if resp.status_code != 200:
            print(f'  [ERROR] HTTP {resp.status_code} on page {page}'); break
        soup = BeautifulSoup(resp.text, 'lxml')
        page_jobs = parse_jobs_page(soup)
        new_jobs = [j for j in page_jobs if j['job_id'] not in seen_ids]
        for j in new_jobs: seen_ids.add(j['job_id'])
        airfrance_jobs.extend(new_jobs)
        print(f'  Page {page}: {len(new_jobs)} new jobs (total: {len(airfrance_jobs)})')
        if not new_jobs:
            consecutive_empty += 1
            if consecutive_empty >= 2: print('  No more pages.'); break
        else: consecutive_empty = 0
        # Check for next page link
        next_link = soup.select_one('a[title*="Next"],a[title*="next"],a[aria-label*="suivante"],a[href*="page="]')
        if not next_link and len(page_jobs) < 30: print('  Last page reached.'); break
        page += 1
        time.sleep(random.uniform(1.0, 2.0))
    except Exception as e:
        print(f'  [ERROR] {e}'); break

# Selenium fallback
if len(airfrance_jobs) < 3:
    print('\n  Falling back to Selenium...')
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    driver = setup_selenium()
    try:
        driver.get(f'{LIST_URL}?LCID=2057'); time.sleep(8)
        for page in range(20):
            soup = BeautifulSoup(driver.page_source, 'lxml')
            page_jobs = parse_jobs_page(soup)
            new_jobs = [j for j in page_jobs if j['job_id'] not in seen_ids]
            for j in new_jobs: seen_ids.add(j['job_id'])
            airfrance_jobs.extend(new_jobs)
            print(f'  Page {page+1}: {len(new_jobs)} new jobs')
            if not new_jobs and page > 0: break
            try:
                btn = driver.find_element(By.CSS_SELECTOR, 'a[title*="Next"],a[aria-label*="next"],a[aria-label*="suivante"]')
                driver.execute_script('arguments[0].click();', btn); time.sleep(3)
            except: break
    except Exception as e:
        print(f'  Selenium error: {e}')
    finally:
        driver.quit()

print(f'\nTotal Air France jobs scraped: {len(airfrance_jobs)}')

AIR FRANCE JOB SCRAPER
Portal: recrutement.airfrance.com (Custom ASP.NET)


  Page 1: 98 new jobs (total: 98)


  Page 2: 68 new jobs (total: 166)


  Page 3: 0 new jobs (total: 166)


  Page 4: 0 new jobs (total: 166)
  No more pages.

Total Air France jobs scraped: 166


In [5]:
df_af = save_results(airfrance_jobs, 'Air_France', OUTPUT_DIR)
if df_af is not None:
    cols = ['title','location_city','seniority_level','business_unit','job_url']
    cols = [c for c in cols if c in df_af.columns]
    print(df_af[cols].head(10).to_string())

  [OK] Saved 81 jobs -> Air_France_jobs_2026-03-31.csv
       Seniority: {'mid': 71, 'lead': 4, 'senior': 3, 'junior': 3}
       Work mode: {'onsite': 81}
       Has JD text: 75/81
       Has job URL: 81/81
       Has business unit: 0/81
                                                                       title location_city seniority_level business_unit                                                                                                                       job_url
0                  Manager équipes Connectivité du Groupe Air France KLM F/H        France            lead                             https://recrutement.airfrance.com/job/job-manager-equipes-connectivite-du-groupe-air-france-klm-f-h-_23574.aspx
4                STAGE – TRANSFORMATION STRATÉGIQUE & PILOTAGE PORTFOLIO F/H        France             mid                                https://recrutement.airfrance.com/job/job-stage-transformation-strategique-pilotage-portfolio-f-h_24320.aspx
6                    